In [1]:
# ======================================================================
# DUAL-ENCODER multi-window UNet -- Jupyter cell.
#
# Two predictor boxes, two encoders, merged at the IMD fine grid:
#   ENCODER A (regional)    : your existing 21 ECMWF vars, current box
#   ENCODER B (large-scale) : OLR + gh200/500/850/1000, bigger box
#                             (-6.5..50 N, 30..120 E) -- captures MJO/BSISO
#                             genesis (equatorward) and the mid-latitude
#                             wave train (poleward) that the regional box
#                             cuts off.
#
# WHY MERGE AT THE FINE GRID, NOT THE BOTTLENECK
# ----------------------------------------------
# The two boxes are different extents on different grids. You cannot concat
# their feature maps directly. Each encoder processes its OWN box at its own
# resolution, then each is grid_sample'd to the shared IMD output grid with
# its OWN sampling grid, and concatenated there. This preserves each
# encoder's spatial reasoning and handles the extent mismatch honestly.
# Merging at the bottleneck (pool -> vector -> concat) would throw away the
# spatial structure that is the whole point of a wider OLR context.
#
# SETUP: run `prepare` once (builds X regional + X_big large-scale, aligned
# to the same inits/windows), then `train`. The OLR/gh zarr must have the
# same (time, step) structure as your ECMWF archive -- you confirmed it does.
#
# ABLATION: this is only meaningful against your single-encoder grad run.
# OLR correlating well ALONE does not guarantee a large gain ON TOP OF tp
# (both track the same convection). The delta vs the single-encoder model
# is the result, small or large.
# ======================================================================

import os, time, json
from contextlib import contextmanager
import numpy as np

# ---------------- CONFIG ----------------
IMD_TARGET_VAR = "rain"
WINDOWS = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]
CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None
TEST_YEARS_N = 3
DTYPE = np.float32

# regional predictors (encoder A) come from your existing ECMWF dataset.
# large-scale predictors (encoder B) come from the OLR/gh dataset.
BIG_VARS = ["top_net_thermal_radiation", "geopotential_height_200",
            "geopotential_height_500", "geopotential_height_850",
            "geopotential_height_1000"]

CACHE = "../data/cache/unet_cache_dual.npz"

class Args:
    cmd = "prepare"
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5

args = Args()
ARG_DICT = {k: getattr(args, k) for k in dir(args) if not k.startswith("_")}
OUT_MAPS = "../results/models/unet_dual.nc"


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ======================================================================
# PREPARE  (two boxes -> two arrays, aligned to the SAME inits/windows)
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


def _prep_box(sub, feature_vars, imd, flat_lat, flat_lon, leads, init):
    """Window-aggregate one dataset's predictors + IMD target. Returns
    stacked-per-window (X, y, doy, wid) lists. IMD target is identical across
    boxes (same valid dates), so we recompute it here but only keep box A's."""
    import xarray as xr
    from dask.diagnostics import ProgressBar
    Xl, yl, dl, wl = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        sel = np.where((leads >= lo) & (leads <= hi))[0]
        if len(sel) == 0:
            raise ValueError(f"no leads in [{lo},{hi}]")
        with ProgressBar():
            Xw = sub.isel(step=sel).mean(dim="step").compute()
        Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)
        vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
        with ProgressBar():
            y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
        ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
        centre = init + np.timedelta64((lo + hi) // 2, "D")
        doya = xr.DataArray(centre, dims="t").dt.dayofyear.values
        Xl.append(Xa); yl.append(ya); dl.append(doya)
        wl.append(np.full(len(Xa), wid, dtype=np.int64))
    return Xl, yl, dl, wl


def _subset_box(ds, imd, pad):
    """Crop a predictor dataset to the padded IMD box (for encoder A) or leave
    it as-is (encoder B, already a chosen box). Sorts ascending lat/lon."""
    ds = ensure_valid_time(ds)
    for c in ("lat", "lon"):
        if ds[c].values[0] > ds[c].values[-1]:
            ds = ds.sortby(c)
    if pad is not None:
        la0, la1 = float(imd.lat.min()), float(imd.lat.max())
        lo0, lo1 = float(imd.lon.min()), float(imd.lon.max())
        ds = ds.sel(lat=slice(la0 - pad, la1 + pad), lon=slice(lo0 - pad, lo1 + pad))
    if MONTHS is not None:
        ds = ds.sel(time=ds["time"].dt.month.isin(list(MONTHS)))
    return ds


def prepare(ecmwf_ds, big_ds, imd_ds, cache_path):
    import numpy as np
    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    with stage("Encoder-A box (regional, padded IMD box)"):
        subA = _subset_box(ecmwf_ds, imd, COARSE_PAD)
        varsA = list(subA.data_vars)
        clatA, clonA = subA["lat"].values, subA["lon"].values
        leadsA = (subA["step"].values / np.timedelta64(1, "D")).astype(int)
        initA = subA["time"].values
        print(f"    A: {dict(subA.sizes)} x {len(varsA)} vars, "
              f"box lat {clatA.min():.1f}..{clatA.max():.1f} "
              f"lon {clonA.min():.1f}..{clonA.max():.1f}")

    with stage("Encoder-B box (large-scale, OLR + gh)"):
        subB = _subset_box(big_ds, imd, pad=None)   # keep its native big box
        missing = [v for v in BIG_VARS if v not in subB.data_vars]
        if missing:
            raise KeyError(f"BIG_VARS missing from big_ds: {missing}")
        varsB = BIG_VARS
        clatB, clonB = subB["lat"].values, subB["lon"].values
        leadsB = (subB["step"].values / np.timedelta64(1, "D")).astype(int)
        initB = subB["time"].values
        print(f"    B: {dict(subB.sizes)} x {len(varsB)} vars, "
              f"box lat {clatB.min():.1f}..{clatB.max():.1f} "
              f"lon {clonB.min():.1f}..{clonB.max():.1f}")

    # --- alignment guards: same inits, same lead availability ---
    if initA.shape != initB.shape or not (initA == initB).all():
        raise ValueError("encoder A and B have different init dates -- cannot "
                         "pair rows. Reindex big_ds onto ecmwf_ds inits first.")
    for lo, hi in [(w[1], w[2]) for w in WINDOWS]:
        if not (((leadsA >= lo) & (leadsA <= hi)).any()
                and ((leadsB >= lo) & (leadsB <= hi)).any()):
            raise ValueError(f"window {lo}-{hi} missing leads in A or B")
    # target box must sit INSIDE both predictor boxes (else grid_sample clamps)
    for nm, (cla, clo) in [("A", (clatA, clonA)), ("B", (clatB, clonB))]:
        if not (cla.min() <= flat_lat.min() and cla.max() >= flat_lat.max()
                and clo.min() <= flat_lon.min() and clo.max() >= flat_lon.max()):
            raise ValueError(f"IMD target grid not fully inside box {nm} "
                             f"(lat {cla.min():.1f}..{cla.max():.1f}, "
                             f"lon {clo.min():.1f}..{clo.max():.1f}) -- "
                             "grid_sample would edge-clamp. widen the box.")

    with stage("Window aggregation (both boxes)"):
        XlA, yl, dl, wl = _prep_box(subA, varsA, imd, flat_lat, flat_lon, leadsA, initA)
        XlB, _, _, _ = _prep_box(subB, varsB, imd, flat_lat, flat_lon, leadsB, initB)
        XA = np.concatenate(XlA); XB = np.concatenate(XlB)
        y = np.concatenate(yl); doy = np.concatenate(dl); wid = np.concatenate(wl)
        year = np.concatenate([initA.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
        print(f"    XA {XA.shape}  XB {XB.shape}  y {y.shape}")
        assert len(XA) == len(XB) == len(y), "A/B/y sample-count mismatch"

    with stage("Mask + test holdout + cache"):
        mask = np.isfinite(y).all(axis=0)
        uy = np.unique(year); test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    strict mask {int(mask.sum())} cells | test {sorted(test_years)}")
        np.savez_compressed(
            cache_path, XA=XA, XB=XB, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test,
            clatA=clatA, clonA=clonA, clatB=clatB, clonB=clonB,
            flat_lat=flat_lat, flat_lon=flat_lon,
            varsA=np.array(varsA), varsB=np.array(varsB),
            window_names=np.array([w[0] for w in WINDOWS]))
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")


def anomalise_fold(Xa_raw, doy, tr):
    """Anomalise + standardise ONE predictor array on train-fold years only."""
    clim = _clim_grid(Xa_raw[tr], doy[tr], CLIM_WINDOW_DAYS)
    A = Xa_raw - clim[doy - 1]
    m = np.nanmean(A[tr], axis=(0, 2, 3), keepdims=True)
    s = np.nanstd(A[tr], axis=(0, 2, 3), keepdims=True)
    s = np.where(s < 1e-8, 1.0, s)
    return np.nan_to_num((A - m) / s).astype(DTYPE)


def anomalise_target(y, doy, tr):
    clim = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)
    return (y - clim[doy - 1]).astype(DTYPE)


# ======================================================================
# MODEL + TRAIN
# ======================================================================

def build_and_run(args, cache_path, out_maps):
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("cuda" if torch.cuda.is_available()
           else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"    device: {dev}")

    z = np.load(cache_path, allow_pickle=False)
    XA, XB, y, doy, wid = z["XA"], z["XB"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clatA, clonA = z["clatA"], z["clonA"]
    clatB, clonB = z["clatB"], z["clonB"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"]); H, W = len(flat_lat), len(flat_lon)
    nA, nB = XA.shape[1], XB.shape[1]

    def make_samp(clat, clon):
        gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
        gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
        gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
        s = np.stack([gxx, gyy], -1).astype(np.float32)[None]
        assert np.abs(s).max() <= 1.0, "fine grid outside a source box"
        return torch.tensor(s).to(dev)

    sampA, sampB = make_samp(clatA, clonA), make_samp(clatB, clonB)

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static = torch.tensor(np.stack([mask.astype(DTYPE),
        np.broadcast_to(lat2, (H, W)).astype(DTYPE),
        np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]).to(dev)

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(s, ci, co, drop=0.0):
            super().__init__()
            L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                 nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                L.append(nn.Dropout2d(drop))
            s.f = nn.Sequential(*L)
        def forward(s, x): return s.f(x)

    class DualUNet(nn.Module):
        """Two coarse encoders (regional A + large-scale B), each grid_sampled
        to the fine grid with its own sampling grid, concatenated there with
        static channels, then a shared UNet decoder."""
        def __init__(s, nA, nB, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            s.emb = nn.Embedding(n_win, emb)
            # encoder A gets the window-id embedding; B is context only
            s.encA1 = Block(nA + emb, base * 2); s.encA2 = Block(base * 2, base * 2)
            s.encB1 = Block(nB, base * 2); s.encB2 = Block(base * 2, base * 2)
            # after grid_sample both give base*2 channels -> merge = base*4, +3 static
            s.inp = Block(base * 4 + 3, base)
            s.d1 = Block(base, base * 2, drop); s.d2 = Block(base * 2, base * 4, drop)
            s.bott = Block(base * 4, base * 4, drop)
            s.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2); s.du2 = Block(base * 4, base * 2, drop)
            s.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2); s.du1 = Block(base * 2, base)
            s.head = nn.Conv2d(base, 1, 1); s.pool = nn.MaxPool2d(2)

        def forward(s, xa, xb, wid, sampA, sampB, static):
            b = xa.shape[0]
            e = s.emb(wid)[:, :, None, None].expand(-1, -1, xa.shape[2], xa.shape[3])
            ca = s.encA2(s.encA1(torch.cat([xa, e], 1)))
            cb = s.encB2(s.encB1(xb))
            fa = F.grid_sample(ca, sampA.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
            fb = F.grid_sample(cb, sampB.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
            f = torch.cat([fa, fb, static.expand(b, -1, -1, -1)], 1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = s.inp(f); e1 = s.d1(s.pool(e0)); e2 = s.d2(s.pool(e1))
            u = s.du2(torch.cat([s.u2(s.bott(e2)), e1], 1))
            u = s.du1(torch.cat([s.u1(u), e0], 1))
            return s.head(u)[:, :, :H0, :W0].squeeze(1)

    def masked_mse(p, t, m):
        return ((p - t) ** 2 * m).sum() / m.sum().clamp(min=1.0)

    def skill_acc(p, t, fin):
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.nanmean(se_m, axis=0)); rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0); pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def train_one(tr_idx, va_idx, XAa, XBa, ya, max_epochs):
        XAt, XBt = torch.tensor(XAa), torch.tensor(XBa)
        yt = torch.tensor(np.nan_to_num(ya)); widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        def dl(idx, sh):
            return DataLoader(TensorDataset(XAt[idx], XBt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)
        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = DualUNet(nA, nB, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xa, xb, wb, yb, mb in tr_dl:
                xa, xb, wb, yb, mb = [t.to(dev) for t in (xa, xb, wb, yb, mb)]
                loss = masked_mse(model(xa, xb, wb, sampA, sampB, static), yb, mb)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            sched.step()
            model.eval(); vl, n = 0.0, 0
            with torch.no_grad():
                for xa, xb, wb, yb, mb in va_dl:
                    xa, xb, wb, yb, mb = [t.to(dev) for t in (xa, xb, wb, yb, mb)]
                    vl += float(masked_mse(model(xa, xb, wb, sampA, sampB, static), yb, mb)) * len(xa)
                    n += len(xa)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, XAa, XBa):
        XAt, XBt, widt = torch.tensor(XAa), torch.tensor(XBa), torch.tensor(wid)
        out = []; model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                out.append(model(XAt[j].to(dev), XBt[j].to(dev), widt[j].to(dev),
                                 sampA, sampB, static).cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    blocks = np.array_split(nontest_years, args.folds)
    resume_path = out_maps.replace(".nc", "_folds.json")
    done = {}
    if os.path.exists(resume_path):
        with open(resume_path) as fh:
            done = {int(k): v for k, v in json.load(fh).items()}
        print(f"    resuming folds {sorted(done)}")

    with stage(f"Rotating-year CV: {args.folds} folds"):
        for fi, val_years in enumerate(blocks):
            if fi in done:
                r = done[fi]; print(f"    fold {fi} (cached): skill {r['skill']:+.3f} | ACC {r['acc']:.3f}"); continue
            vy = set(val_years.tolist())
            va_i = np.isin(year, list(vy)) & ~is_test
            tr_i = ~np.isin(year, list(vy)) & ~is_test
            XAa = anomalise_fold(XA, doy, tr_i)
            XBa = anomalise_fold(XB, doy, tr_i)
            ya = anomalise_target(y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0], XAa, XBa, ya, args.epochs)
            p = predict(model, np.where(va_i)[0], XAa, XBa)
            t = ya[va_i]; fin = np.isfinite(y[va_i]) & mask[None]
            sk, ac = skill_acc(p, t, fin)
            ms, ma = float(np.nanmean(sk[mask])), float(np.nanmean(ac[mask]))
            done[fi] = {"skill": ms, "acc": ma, "val_years": sorted(vy), "epochs": int(eps)}
            with open(resume_path, "w") as fh:
                json.dump({str(k): v for k, v in done.items()}, fh, indent=2)
            print(f"    fold {fi} val {sorted(vy)}: skill {ms:+.3f} | ACC {ma:.3f} | {eps} ep   [saved]", flush=True)
        ks = [i for i in range(args.folds) if i in done]
        fs = [done[i]["skill"] for i in ks]; fa = [done[i]["acc"] for i in ks]
        print(f"\n    [DUAL] CV skill {np.mean(fs):+.4f} +/- {np.std(fs):.4f} | "
              f"CV ACC {np.mean(fa):.4f} +/- {np.std(fa):.4f}")

    # ---------- final model -> test ----------
    with stage("Final model -> test"):
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = (~is_test) & ~va_i
        XAa = anomalise_fold(XA, doy, fit_i)
        XBa = anomalise_fold(XB, doy, fit_i)
        ya = anomalise_target(y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0], XAa, XBa, ya, args.epochs)
        te_i = np.where(is_test)[0]
        p = predict(model, te_i, XAa, XBa); t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]
        import xarray as xr
        wid_te = wid[is_test]; dvars = {}
        print(f"    trained {eps} ep")
        for w in range(n_win):
            sm = wid_te == w
            if sm.sum() == 0: continue
            sk, ac = skill_acc(p[sm], t[sm], fin[sm])
            wn = str(z["window_names"][w])
            dvars[f"{wn}_skill"] = (("lat", "lon"), sk); dvars[f"{wn}_acc"] = (("lat", "lon"), ac)
            print(f"    test {wn:>8}: skill {np.nanmean(sk[mask]):+.4f} | "
                  f"ACC {np.nanmean(ac[mask]):.4f} | {100*np.nanmean(sk[mask]>0):.0f}% cells+")
        out = xr.Dataset(dvars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["model"] = "dual-encoder (regional + OLR/gh large-scale)"
        out.to_netcdf(out_maps)
        np.savez_compressed(out_maps.replace(".nc", "_pred.npz"),
                            pred=p, obs=t, wid=wid[is_test])
        torch.save({"state": model.state_dict(), "args": ARG_DICT}, out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ _pred.npz, .pt)")


# ======================================================================
if args.cmd == "prepare":
    prepare(ds_ecmv, ds_big, ds_imd, CACHE)          # noqa: F821
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f"run prepare first ({CACHE} missing)")
    build_and_run(args, CACHE, OUT_MAPS)

NameError: name 'ds_ecmv' is not defined